# Train Activity Text Model

Notebook version of scripts/train_activity_text_model.py with visible outputs.

In [4]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.pipeline import FeatureUnion, Pipeline

ROOT = Path.cwd().resolve().parents[0]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.activity_text_model import (
    ACTIVITY_PROFILE_COLS,
    ACTIVITY_PROTOTYPES,
    TRAINING_PHRASES,
    normalize_text,
)

RANDOM_STATE = 42

In [5]:
def expand_phrase(phrase: str) -> list[str]:
    normalized = normalize_text(phrase)
    variants = {
        normalized,
        f"{normalized} ahora",
        f"voy a {normalized}",
        f"quiero {normalized}",
        f"tengo que {normalized}",
        f"necesito {normalized}",
        f"me apetece {normalized}",
        f"estoy para {normalized}",
        f"me toca {normalized}",
        f"hora de {normalized}",
        f"plan de {normalized}",
        f"vamos a {normalized}",
    }
    return sorted(variants)

In [6]:
def build_training_frame() -> pd.DataFrame:
    rows = []
    for activity_name, phrases in TRAINING_PHRASES.items():
        for phrase in phrases:
            for text in expand_phrase(phrase):
                row = {
                    "text": normalize_text(text),
                    "activity_name": activity_name,
                }
                row.update(dict(zip(ACTIVITY_PROFILE_COLS, ACTIVITY_PROTOTYPES[activity_name])))
                rows.append(row)
    return pd.DataFrame(rows)


def train_models(df: pd.DataFrame) -> dict[str, Pipeline]:
    regressor = Pipeline(
        steps=[
            (
                "features",
                FeatureUnion(
                    [
                        ("word", TfidfVectorizer(ngram_range=(1, 2), min_df=1)),
                        ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=1)),
                    ]
                ),
            ),
            ("regressor", KNeighborsRegressor(n_neighbors=3, weights="distance", metric="cosine")),
        ]
    )
    classifier = Pipeline(
        steps=[
            (
                "features",
                FeatureUnion(
                    [
                        ("word", TfidfVectorizer(ngram_range=(1, 2), min_df=1)),
                        ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=1)),
                    ]
                ),
            ),
            ("classifier", KNeighborsClassifier(n_neighbors=3, weights="distance", metric="cosine")),
        ]
    )
    regressor.fit(df["text"], df[ACTIVITY_PROFILE_COLS])
    classifier.fit(df["text"], df["activity_name"])
    return {"regressor": regressor, "classifier": classifier}


def save_model(model: dict[str, Pipeline], df: pd.DataFrame) -> None:
    models_dir = ROOT / "models"
    models_dir.mkdir(parents=True, exist_ok=True)
    joblib.dump(model, models_dir / "activity_text_interpreter.joblib")

    metadata = {
        "created_at": datetime.now(timezone.utc).isoformat(),
        "random_state": RANDOM_STATE,
        "model_type": "TFIDF word/char ngrams + KNeighbors regressor + KNeighbors classifier",
        "activity_profile_columns": ACTIVITY_PROFILE_COLS,
        "activity_prototypes": ACTIVITY_PROTOTYPES,
        "training_examples": int(len(df)),
        "note": (
            "Este modelo combina un interprete linguistico con reglas por palabras clave. "
            "Convierte texto libre en dimensiones musicales como energia, movimiento, foco y calma."
        ),
    }
    (models_dir / "activity_text_interpreter_metadata.json").write_text(
        json.dumps(metadata, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    df.to_csv(ROOT / "data_lake" / "recommender" / "activity_text_training_examples.csv", index=False)


def debug_examples(model: dict[str, Pipeline]) -> None:
    examples = [
        "quiero fregar el suelo",
        "estoy triste y voy a limpiar la cocina",
        "quiero llorar",
        "voy a programar un rato",
        "salgo a correr",
    ]
    normalized_examples = [normalize_text(example) for example in examples]
    predictions = model["regressor"].predict(normalized_examples)
    labels = model["classifier"].predict(normalized_examples)
    for example, values, label in zip(examples, predictions, labels):
        pretty = dict(zip(ACTIVITY_PROFILE_COLS, values.clip(0.0, 1.0).round(3).tolist()))
        print(example, label, pretty)

In [7]:
print("Building training data...")
df = build_training_frame()
print(f"Training rows: {len(df)}")
df.head()

Building training data...
Training rows: 1740


,text,activity_name,activity_movement,activity_energy,activity_positivity,activity_focus,activity_calm,activity_acoustic
0,estoy para quiero llorar,desahogo_emocional,0.0,0.05,0.05,0.3,0.95,0.85
1,hora de quiero llorar,desahogo_emocional,0.0,0.05,0.05,0.3,0.95,0.85
2,me apetece quiero llorar,desahogo_emocional,0.0,0.05,0.05,0.3,0.95,0.85
3,me toca quiero llorar,desahogo_emocional,0.0,0.05,0.05,0.3,0.95,0.85
4,necesito quiero llorar,desahogo_emocional,0.0,0.05,0.05,0.3,0.95,0.85


In [8]:
print("Training models...")
model = train_models(df)
print("Saving model artifacts...")
save_model(model, df)
print("Debugging examples...")
debug_examples(model)

Training models...
Saving model artifacts...
Debugging examples...
quiero fregar el suelo limpieza_domestica {'activity_movement': 0.75, 'activity_energy': 0.7, 'activity_positivity': 0.55, 'activity_focus': 0.35, 'activity_calm': 0.15, 'activity_acoustic': 0.2}
estoy triste y voy a limpiar la cocina limpieza_domestica {'activity_movement': 0.75, 'activity_energy': 0.7, 'activity_positivity': 0.55, 'activity_focus': 0.35, 'activity_calm': 0.15, 'activity_acoustic': 0.2}
quiero llorar desahogo_emocional {'activity_movement': 0.0, 'activity_energy': 0.05, 'activity_positivity': 0.05, 'activity_focus': 0.3, 'activity_calm': 0.95, 'activity_acoustic': 0.85}
voy a programar un rato estudio_trabajo {'activity_movement': 0.15, 'activity_energy': 0.25, 'activity_positivity': 0.45, 'activity_focus': 0.95, 'activity_calm': 0.7, 'activity_acoustic': 0.75}
salgo a correr correr {'activity_movement': 1.0, 'activity_energy': 0.95, 'activity_positivity': 0.7, 'activity_focus': 0.35, 'activity_calm': 